# JWT Inbound Auth 및 OAuth 보호 Gateway를 사용하는 Harness

## 중요한 이유

프로덕션 에이전트는 독립적으로 실행되지 않습니다. 인증이 필요한 실제 사용자에게
서비스를 제공하고, 별도의 자격 증명이 필요한 다운스트림 API를 호출합니다.
이를 올바르게 구현하면 다음을 보장할 수 있습니다.

- **권한이 있는 사용자만 에이전트 호출** - 엔드포인트 URL을 아는 누구나 호출할 수 없도록 제한
- **에이전트가 올바른 자격 증명으로 도구 호출** - 코드에 보안 정보를 넣지 않고
  각 요청에 백엔드용 적절한 토큰 포함
- **보안 정보를 안전하게 관리** - 노트북에 클라이언트 보안 암호를, 환경 변수에 토큰을 저장하지 않음

AgentCore Harness는 설정 수준의 두 기본 요소인 **inbound auth**
(Harness를 호출할 수 있는 주체)와 **outbound auth**(Harness가 도구를 호출하는 방법)로
이 문제를 해결합니다. 이 노트북에서는 두 방식을 모두 살펴봅니다.

## AgentCore Harness란 무엇인가요?

에이전트 설정을 실행 중인 에이전트로 전환하는 관리형 컴퓨팅 환경입니다.
모델, 시스템 프롬프트, 도구 및 인증을 선언하면 AgentCore가 오케스트레이션 루프,
컴퓨팅(격리된 Firecracker microVM), 도구 호출, Memory 및 Observability를
처리합니다. 프레임워크 코드, 컨테이너 또는 배포 파이프라인이 필요하지 않습니다.

전체 문서는 [AgentCore harness Developer Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/harness.html)를
참조하세요.

## 학습 내용

이 노트북에서는 두 가지 Harness 기능에 중점을 둡니다.

**Inbound auth(`CUSTOM_JWT`)** - harness가 요청을 수락하기 전에 호출자가 유효한 JWT를
제시하도록 요구합니다. 여기서는 Cognito user pool을 사용하지만 모든 OIDC 공급자를
사용할 수 있습니다. 이 방식으로 에이전트 endpoint를 보호합니다.

**Outbound auth(`outboundAuth.oauth`)** - harness가 AgentCore Gateway에 인증하기 위한
OAuth 토큰(client credentials grant)을 자동으로 가져옵니다. 자격 증명 공급자는
한 번만 등록하며, 이후 Harness가 도구를 호출할 때마다 토큰 교환을 처리합니다.
호출 요청에 보안 정보를 포함할 필요가 없습니다.

## 아키텍처

<img src="images/architecture.jpg" alt="아키텍처" width="800"/>

## 사전 요구 사항

- Bedrock AgentCore에 접근할 수 있는 AWS 계정
- AWS 자격 증명 설정(`aws configure` 또는 환경 변수)
- Python 3.10+, `boto3 >= 1.42.80`, `requests`, `jupyter`
- Bedrock 모델 접근 활성화

## 프로젝트 구조

```

├── harness_oauth_gateway.ipynb   ← 이 노트북
├── utils/
│   ├── setup_helpers.py          ← 인프라 설정 및 정리
│   └── lambda_function_code.py   ← Lambda 핸들러
├── requirements.txt
└── README.md
```

인프라 설정은 `utils/setup_helpers.py`에 있습니다. 이 노트북은 Harness에
중점을 둡니다. 모든 셀은 멱등성을 보장하므로 안전하게 다시 실행할 수 있습니다.

## 0단계: 모듈 가져오기 및 설정

In [ ]:
import boto3
import json
import time
import uuid
import getpass
import requests as http_requests
import urllib.parse
from utils.setup_helpers import (
    create_user_auth_pool,
    create_m2m_pool,
    create_credential_provider,
    deploy_lambda,
    create_gateway_with_lambda_target,
    create_harness_execution_role,
    cleanup_all,
)

REGION = boto3.session.Session().region_name or "us-east-1"
ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
ac_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)

PREFIX = "harness-oauth-demo"
print(f"Region: {REGION}  Account: {ACCOUNT_ID}")

## 1단계: 인프라 프로비저닝

각 헬퍼는 하나의 리소스 그룹을 생성하고 ID/ARN을 반환합니다.
모든 헬퍼는 기존 리소스를 먼저 확인하므로 안전하게 다시 실행할 수 있습니다.
전체 구현은 `utils/setup_helpers.py`를 참조하세요.

| 헬퍼 | 생성되는 리소스 |
|--------|----------------|
| `create_user_auth_pool` | Cognito User Auth Pool - user pool, 앱 클라이언트(USER_PASSWORD_AUTH), 테스트 사용자 |
| `create_m2m_pool` | Cognito M2M Pool - user pool, 리소스 서버 + 범위, 도메인, 앱 클라이언트(client_credentials) |
| `create_credential_provider` | M2M Pool을 가리키는 AgentCore Identity의 OAuth2 자격 증명 공급자 |
| `deploy_lambda` | IAM 역할 + `utils/lambda_function_code.py`의 Lambda function |
| `create_gateway_with_lambda_target` | Gateway(CUSTOM_JWT → M2M Pool) + Lambda 대상(GATEWAY_IAM_ROLE) |
| `create_harness_execution_role` | Bedrock, Gateway, Token Vault, CloudWatch, X-Ray 권한이 있는 IAM 역할 |

### 1a. Cognito User Auth Pool - User Auth
Harness를 호출하는 최종 사용자를 인증하는 풀을 생성합니다.

In [ ]:
print("Enter credentials for the test user in User Auth Pool:")
USER1_NAME = getpass.getpass("Username: ")
USER1_PASS = getpass.getpass("Password (min 8 chars): ")

pool1 = create_user_auth_pool(REGION, PREFIX, USER1_NAME, USER1_PASS)
print(f"\nDiscovery URL: {pool1['discovery_url']}")

### 1b. Cognito M2M Pool - M2M
시스템 간 인증을 위한 풀을 생성합니다(사용자 지정 범위가 있는 client credentials grant).

In [ ]:
pool2 = create_m2m_pool(REGION, PREFIX)
print(f"\nScope: {pool2['scope']}")
print(f"Discovery URL: {pool2['discovery_url']}")

### 1c. OAuth2 Credential Provider
Harness가 outbound gateway auth를 위한 M2M 토큰을 가져올 수 있도록
M2M Pool의 클라이언트 자격 증명을 AgentCore Identity에 등록합니다.

In [ ]:
cred = create_credential_provider(
    REGION,
    PREFIX,
    discovery_url=pool2["discovery_url"],
    client_id=pool2["client_id"],
    client_secret=pool2["client_secret"],
)

### 1d. Lambda Function
`utils/lambda_function_code.py`의 주문 관리 Lambda를 배포합니다.

In [ ]:
lam = deploy_lambda(REGION, PREFIX)

### 1e. Gateway + Lambda Target
CUSTOM_JWT inbound auth(M2M Pool)를 사용하는 gateway를 생성하고,
GATEWAY_IAM_ROLE outbound auth를 사용하는 Lambda를 대상으로 추가합니다.

In [ ]:
gw = create_gateway_with_lambda_target(
    REGION,
    PREFIX,
    ACCOUNT_ID,
    discovery_url=pool2["discovery_url"],
    allowed_client=pool2["client_id"],
    allowed_scope=pool2["scope"],
    lambda_arn=lam["function_arn"],
    lambda_function_name=lam["function_name"],
)
print(f"\nGateway ARN: {gw['gateway_arn']}")

### 1f. Harness 실행 역할
런타임에 Harness가 수임하는 IAM 역할을 생성합니다. 이 역할에는 Bedrock, Gateway,
OAuth2 Token Vault, Secrets Manager 및 관찰성 권한이 포함됩니다.

In [ ]:
harness_role = create_harness_execution_role(REGION, PREFIX, ACCOUNT_ID)

## 2단계: CUSTOM_JWT Inbound Auth를 사용하는 Harness 생성

이 노트북의 핵심 단계입니다. 다음과 같이 Harness를 생성합니다.

- **Inbound auth**: **User Auth Pool**을 가리키는 `CUSTOM_JWT` 권한 부여자. 호출자는
  Harness를 호출할 때 User Auth Pool의 유효한 JWT를 제시해야 합니다.
- **Model**: Amazon Bedrock을 통한 Claude Sonnet 4.
- **도구**: 자격 증명 공급자(M2M Pool client credentials)와 `outboundAuth.oauth`를
  사용하는 AgentCore Gateway. Harness가 Gateway 인증용 M2M 토큰을 자동으로 가져옵니다.
- **시스템 프롬프트**: 주문 관리 어시스턴트.

이전 실행에서 생성한 Harness가 이미 있으면 재사용합니다.

In [ ]:
HARNESS_NAME = f"{PREFIX}-harness".replace("-", "_")

# 생성을 시도하고 이미 존재하면 조회
try:
    harness_resp = ac_control.create_harness(
        harnessName=HARNESS_NAME,
        executionRoleArn=harness_role["role_arn"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": pool1["discovery_url"],
                "allowedClients": [pool1["client_id"]],
            }
        },
        model={
            "bedrockModelConfig": {
                "modelId": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
            }
        },
        systemPrompt=[
            {
                "text": (
                    "You are an order management assistant. "
                    "Use the gateway tools to look up and update orders. "
                    "Always confirm the order details before making changes."
                )
            }
        ],
        tools=[
            {
                "type": "agentcore_gateway",
                "name": "order-gateway",
                "config": {
                    "agentCoreGateway": {
                        "gatewayArn": gw["gateway_arn"],
                        "outboundAuth": {
                            "oauth": {
                                "providerArn": cred["arn"],
                                "scopes": [pool2["scope"]],
                                "grantType": "CLIENT_CREDENTIALS",
                            }
                        },
                    }
                },
            }
        ],
    )
    HARNESS_ID = harness_resp["harness"]["harnessId"]
    HARNESS_ARN = harness_resp["harness"]["arn"]
    print(f"Harness created: {HARNESS_ID}")
except ac_control.exceptions.ConflictException:
    HARNESS_ID = None
    for h in ac_control.list_harnesses().get("harnesses", []):
        if h.get("harnessName") == HARNESS_NAME:
            HARNESS_ID = h["harnessId"]
            HARNESS_ARN = h["arn"]
            break
    if not HARNESS_ID:
        raise RuntimeError(f"Harness {HARNESS_NAME} conflict but not found")
    print(f"Harness already exists: {HARNESS_ID}")

print(f"Harness ID:  {HARNESS_ID}")
print(f"Harness ARN: {HARNESS_ARN}")

print("Waiting for harness to become READY...")
for _ in range(30):
    h_status = ac_control.get_harness(harnessId=HARNESS_ID)["harness"]["status"]
    if h_status == "READY":
        break
    time.sleep(10)
print(f"Harness status: {h_status}")

## 3단계: User Auth Pool에서 Bearer 토큰 가져오기

`USER_PASSWORD_AUTH`로 테스트 사용자를 인증하여 JWT 액세스 토큰을 가져옵니다.
Harness를 호출할 때 이 토큰을 `Authorization: Bearer <token>`으로
전달합니다.

In [ ]:
auth_result = cognito.initiate_auth(
    ClientId=pool1["client_id"],
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": USER1_NAME, "PASSWORD": USER1_PASS},
)
BEARER_TOKEN = auth_result["AuthenticationResult"]["AccessToken"]
print(f"Got bearer token (first 20 chars): {BEARER_TOKEN[:20]}...")

## 4단계: Bearer 토큰으로 Harness 호출

boto3는 bearer-token 호출을 지원하지 않으므로 HTTPS 엔드포인트를 직접
호출합니다. 흐름은 다음과 같습니다.

1. Harness가 User Auth Pool에서 JWT 검증(inbound auth)
2. 에이전트가 사용자 메시지를 추론하고 Gateway 도구 호출 결정
3. Harness가 OAuth2 자격 증명 공급자를 사용하여 M2M Pool에서 M2M 토큰 획득
4. Harness가 해당 M2M 토큰으로 Gateway 호출(outbound auth)
5. Gateway가 M2M Pool에서 M2M 토큰을 검증하고 Lambda 호출
6. 결과가 에이전트를 통해 사용자에게 반환

In [ ]:
escaped_arn = urllib.parse.quote(HARNESS_ARN, safe="")
url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/harnesses/invoke?harnessArn={escaped_arn}"
SESSION_ID = f"notebook-session-{uuid.uuid4().hex}"

headers = {
    "Authorization": f"Bearer {BEARER_TOKEN}",
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": SESSION_ID,
}

payload = {
    "messages": [
        {
            "role": "user",
            "content": [{"text": "Look up order ORD-001 and tell me its status."}],
        }
    ],
}

print(f"Session: {SESSION_ID}")
print(f"URL: {url[:80]}...")
print()

# Harness는 일반 JSON이 아닌 AWS event-stream(binary framed protocol)을 반환
# 응답을 스트리밍하고 이벤트 페이로드에서 텍스트 델타 추출
resp = http_requests.post(url, headers=headers, json=payload, timeout=120, stream=True)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    # event-stream 구문 분석: 각 이벤트의 binary frame에 JSON 페이로드가 포함됨
    # 원시 바이트 스트림에서 JSON 객체 추출
    full_text = []
    raw = resp.content
    # 바이너리 스트림에서 모든 JSON 객체 찾기
    import re

    json_objects = re.findall(rb"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", raw)
    for obj_bytes in json_objects:
        try:
            obj = json.loads(obj_bytes.decode("utf-8", errors="ignore"))
            # contentBlockDelta 이벤트에 텍스트가 포함됨
            delta = obj.get("delta", {})
            if "text" in delta:
                full_text.append(delta["text"])
                print(delta["text"], end="", flush=True)
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue
    print()  # 스트리밍 후 줄 바꿈
    if full_text:
        print(f"\n--- Full response ({len(''.join(full_text))} chars) ---")
    else:
        print("\n(No text deltas found in stream. Raw first 500 bytes below)")
        print(repr(raw[:500]))
else:
    print(f"Error {resp.status_code}: {resp.text[:1000]}")

### 방금 처리된 작업

한 번의 요청으로 전체 인증 체인이 실행되었습니다.

1. **User Auth Pool JWT** 전송 → Harness가 검증(inbound auth)
2. 에이전트가 `get_order` 호출 결정 → Harness가 자격 증명 공급자를 통해
   **M2M Pool의 M2M 토큰** 획득(outbound auth)
3. Harness가 M2M 토큰으로 **Gateway** 호출 → Gateway가 토큰을 검증하고
   자체 IAM 역할로 **Lambda** 호출
4. 주문 세부 정보가 에이전트를 통해 사용자에게 반환

세 가지 인증 메커니즘을 사용하면서 호출에 포함된 보안 정보는 없습니다. Harness는
`outboundAuth.oauth` 설정에 따라 토큰 교환을 자동으로 처리했습니다.

Harness 보안에 관한 자세한 내용은 [AgentCore Harness Developer Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/harness.html)를
참조하세요.

## 5단계: 리소스 정리

모든 리소스를 역순으로 삭제합니다. 이름으로 리소스를 검색하므로 커널을
다시 시작한 뒤에도 작동합니다. 리소스가 없으면 문제없이 건너뜁니다.

In [ ]:
cleanup_all(REGION, PREFIX)